In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, silver

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
silver.ensure_tables(spark, cfg)
tables = silver.load_silver()
print(cfg["code"], env, len(tables), "tables")

In [0]:
start = time.time()
total = {"rows": 0, "quarantined": 0, "inserted": 0, "updated": 0}
for table, spec in tables.items():                    # config order = parents before children
    t0 = time.time()
    r = silver.load_table(spark, cfg, table, spec)
    for k in total:
        total[k] += r[k]
    print(f"{table:18} {r}  {time.time() - t0:.0f}s")
for name, m in silver.load_id_maps().items():         # same entity under several ids -> one survivor
    t0 = time.time()
    print(f"{name:18} {silver.build_id_map(spark, cfg, name, m)}  {time.time() - t0:.0f}s")
if os.path.exists(os.path.join(os.path.dirname(os.getcwd()), "src", "lh", "recon.py")):
    from lh import recon                              # only the lending company has reconciliations
    t0 = time.time()
    print(f"{'reconciliation':18} {recon.run(spark, cfg)}  {time.time() - t0:.0f}s")
print("TOTAL", total, f"in {time.time() - start:.0f}s")

passed = silver.checkpoint(spark, cfg, tables)        # the gate before gold
common.log_run(spark, cfg, "silver", "success" if passed else "failed",
               total["inserted"] + total["updated"], str(total))
if not passed:
    raise RuntimeError("Silver checkpoint failed: gold must not run on this data")